In [1]:
import pandas as pd
import plotly.express as px
from scipy.stats import spearmanr

from utils import get_recession_data, get_recession_start_end_list

In [2]:
fed_funds = pd.read_csv("data/fed_funds.csv").dropna()
fed_funds["date"] = pd.to_datetime(fed_funds["date"])
fed_funds.set_index("date", inplace=True)
fed_funds

,interest rate
date,
1954-07-01,0.80
1954-08-01,1.22
1954-09-01,1.07
1954-10-01,0.85
1954-11-01,0.83
...,...
2023-11-01,5.33
2023-12-01,5.33
2024-01-01,5.33


In [3]:
gold = pd.read_csv("data/WPU10210501.csv").dropna()
gold["date"] = pd.to_datetime(gold["date"])
gold = gold.set_index("date")
gold

,gold price
date,
1985-06-01,100.000
1985-07-01,100.200
1985-08-01,100.200
1985-09-01,99.000
1985-10-01,99.300
...,...
2021-08-01,445.696
2021-09-01,445.591
2021-10-01,442.904


In [4]:
# Ensure df1 and df2 have the same dates in their indices
common_dates = gold.index.intersection(fed_funds.index)
fed_funds = fed_funds.reindex(common_dates)
gold = gold.reindex(common_dates)

if not fed_funds.index.equals(gold.index):
    raise ValueError("DataFrames do not have the same dates in their indices")

In [5]:
df = pd.merge(
    fed_funds["interest rate"], gold["gold price"], left_index=True, right_index=True
)

In [6]:
df.corr("spearman")

,interest rate,gold price
interest rate,1.00000,-0.76052
gold price,-0.76052,1.00000


In [7]:
corr_df = df["interest rate"].rolling(window=60).corr(df["gold price"])
corr_df

date
1985-06-01         NaN
1985-07-01         NaN
1985-08-01         NaN
1985-09-01         NaN
1985-10-01         NaN
                ...   
2021-08-01   -0.662983
2021-09-01   -0.684221
2021-10-01   -0.708095
2021-11-01   -0.738117
2021-12-01   -0.766848
Length: 381, dtype: float64

In [8]:
# normalize each row of data in df except for the index and correlation
df = (df - df.min()) / (df.max() - df.min())

df["correlation"] = corr_df
df = df.dropna()
df.reset_index(inplace=True)

recession_df = get_recession_data()
recession_df = recession_df.reindex(common_dates)

In [9]:
fig = px.line(
    df,
    x="date",
    y=df.columns,
    hover_data={"date": "|%B %d, %Y"},
    title="Correlation between Gold Prices and Interest Rates",
    template="plotly_dark",
    width=1200,
    height=600,
)

for row in get_recession_start_end_list(recession_df):
    x0 = str(row[0].date())
    x1 = str(row[1].date())

    fig.add_vrect(
        x0=x0,
        x1=x1,
        fillcolor="red",
        opacity=0.25,
        line_width=0,
    )

fig.show()

In [10]:
"""Double check the correlation using a different method"""

spear_corr, _ = spearmanr(df["gold price"], df["interest rate"])
print("Spearmans correlation: %.2f" % spear_corr)

Spearmans correlation: -0.77


In [11]:
# TODO: what is the corr between gold and silver?
silver = pd.read_csv("data/SilverFutures.csv").dropna()
silver["date"] = pd.to_datetime(silver["date"])
silver = silver.set_index("date")
silver = silver.iloc[::-1]  # put the data in ascending order
silver

,price,open,high,low,vol,change %
date,,,,,,
1980-02-01,36.340,35.000,39.000,34.000,2.73K,0.94%
1980-03-01,19.350,36.100,37.600,19.350,19.72K,-46.75%
1980-04-01,13.000,18.350,18.350,13.000,3.32K,-32.82%
1980-05-01,14.060,12.000,14.060,10.900,15.36K,8.15%
1980-06-01,16.900,14.875,17.800,14.875,3.78K,20.20%
...,...,...,...,...,...,...
2023-12-01,24.086,25.525,26.050,22.700,85.51K,-4.77%
2024-01-01,23.051,23.845,24.070,21.925,1.17K,-4.30%
2024-02-01,22.666,22.875,23.195,22.005,7.16K,-1.67%


In [12]:
gold = pd.read_csv("data/WPU10210501.csv").dropna()
gold["date"] = pd.to_datetime(gold["date"])
gold = gold.set_index("date")
gold

,gold price
date,
1985-06-01,100.000
1985-07-01,100.200
1985-08-01,100.200
1985-09-01,99.000
1985-10-01,99.300
...,...
2021-08-01,445.696
2021-09-01,445.591
2021-10-01,442.904


In [13]:
common_dates = gold.index.intersection(silver.index)
silver = silver.reindex(common_dates)
gold = gold.reindex(common_dates)

if not silver.index.equals(gold.index):
    raise ValueError("DataFrames do not have the same dates in their indices")

In [14]:
df = pd.merge(silver["price"], gold["gold price"], left_index=True, right_index=True)

In [15]:
df.corr("spearman")

,price,gold price
price,1.000000,0.908787
gold price,0.908787,1.000000
